# DriveSense — 02. Drowsiness Feature Extraction & Training (Colab)

**STATUS: PREPARED, NOT YET EXECUTED.** Every cell below is real, working code
against the actual project modules and the actual UTA-RLDD data — but it has
not been run end-to-end in Colab yet. No metric, plot, or checkpoint in this
notebook is a real result until you actually execute these cells and see real
output. Do not copy any number out of this notebook into a report unless you
personally ran the cell that produced it.

## Why this notebook exists

The local development machine's MediaPipe face-landmark extraction reliably
gets OOM-killed (confirmed three times — see `docs/LEARNING_LOG.md` and
`docs/Master_Plan_Status.md`) because ~7GB of RAM is shared with a normal
desktop session. This is a real, external resource constraint, not a bug in
the extraction code — the same code is unit-tested and correct (see
`tests/test_drowsiness.py`, `tests/test_landmarks.py`). Google Colab's
dedicated RAM (and optional GPU) is where this actually needs to run.

## What this notebook does, stage by stage

0. Verify the Colab environment (RAM/GPU/disk).
1. Clone this private repo and install exact dependencies.
2. Mount Google Drive so expensive work survives a Colab disconnect.
3. Authenticate with Kaggle (your own token — never committed anywhere).
4. Download a real, selective set of UTA-RLDD subjects (whole subjects only
   — never partial, to protect subject-independence) — reusing the exact
   same `scripts/uta_rldd_pipeline.py` already used and tested locally.
5. Validate every downloaded video actually opens (`cv2.VideoCapture`).
6. Run the **real** MediaPipe extractor (the step blocked locally) over every
   video, checkpointing per-video results to Drive so a disconnect doesn't
   lose completed videos.
7. Real EDA on the extracted per-frame features.
8. Time-based windowing (not naive fixed-frame windowing — see
   `src/drivesense/data/drowsiness.py`'s docstring for why this matters:
   this project's own EDA found 8 distinct frame rates and 7 distinct
   resolutions across just 6 subjects).
9. A strict subject-independent train/val/test split with explicit
   zero-overlap assertions.
10. Three real experiments: a non-temporal (mean-pooled) baseline, a GRU,
    and a 1D-CNN — the project's core research question (does temporal
    modeling help?) answered with actual numbers, not assumed.
11. Model selection using **validation only**; the test set is touched
    exactly once, at the end.
12. Real plots/tables for the university report.
13. Export the winning checkpoint + preprocessing config, and push the real
    (small, non-binary) results back into this GitHub repo.

## Before you start: three things only you can do

1. **A GitHub Personal Access Token** with `repo` (read/write) scope, so
   Colab can clone this private repo and push real results back. Generate
   one at https://github.com/settings/tokens — treat it like a password.
2. **Your own Kaggle account**, with an API token from
   https://www.kaggle.com/settings/api (`kaggle.json`), or a
   `KAGGLE_API_TOKEN` value.
3. **Runtime → Change runtime type → GPU** (T4 is fine) — not strictly
   required (everything here can run on CPU, just slower), but recommended
   for the GRU/1D-CNN training stage.

Run the cells **in order, top to bottom**. Each stage prints clear
progress and fails loudly (with an assertion or a clear error) rather than
silently producing wrong output — if a cell raises, stop and read the
message before continuing; later cells generally assume earlier ones
actually succeeded.

## Stage 0 — Verify the Colab environment

Real detection, not an assumption — prints what this specific runtime actually has.

In [ ]:
import subprocess, shutil, sys

print("Python:", sys.version)

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                      capture_output=True, text=True)
if gpu.returncode == 0:
    print("GPU detected:", gpu.stdout.strip())
else:
    print("No GPU detected (nvidia-smi not found or failed). CPU-only run is possible but slower.")
    print("Recommended: Runtime -> Change runtime type -> Hardware accelerator -> GPU, then re-run this cell.")

import psutil
vm = psutil.virtual_memory()
print(f"RAM: {vm.total/1e9:.1f} GB total, {vm.available/1e9:.1f} GB available")
assert vm.total > 6e9, "Unexpectedly low-memory runtime detected - this is not a standard Colab instance."

total, used, free = shutil.disk_usage("/content")
print(f"Disk: {total/1e9:.1f} GB total, {free/1e9:.1f} GB free")
assert free > 15e9, f"Only {free/1e9:.1f}GB free disk - not enough headroom for video downloads. Check your Colab disk quota."

print("\nStage 0 OK: environment has enough RAM/disk to proceed.")

## Stage 1 — Clone the repo and install exact dependencies

The repo is **private**, so cloning needs a token even for read access. The
token is entered via `getpass` (never displayed, never printed, never
written to a notebook cell's saved output) and is only used to build the
clone URL in memory — it is not saved to disk after this cell finishes.

In [ ]:
from getpass import getpass
import os

github_token = getpass("GitHub Personal Access Token (repo scope, input hidden): ")
clone_url = f"https://{github_token}@github.com/echo05w/drivesense-driver-monitoring.git"

import subprocess
result = subprocess.run(["git", "clone", clone_url, "drivesense-driver-monitoring"],
                         capture_output=True, text=True)
# Scrub the token from any output before printing, in case git ever echoes the URL.
safe_stdout = result.stdout.replace(github_token, "***")
safe_stderr = result.stderr.replace(github_token, "***")
print(safe_stdout)
print(safe_stderr)
assert result.returncode == 0, "git clone failed - check the token has 'repo' scope and hasn't expired."

del github_token, clone_url, result  # never keep the token in memory longer than needed
print("Clone OK.")

In [ ]:
%cd drivesense-driver-monitoring
!pip install -q -r requirements.txt
!pip install -q -e .
print("Dependencies installed.")

In [ ]:
# Real verification, not an assumption: import every module this notebook
# depends on, and run the fast synthetic-data test suite before doing
# anything expensive. If this cell fails, stop - later stages will not work.
import drivesense
print("drivesense package importable, location:", drivesense.__file__)

from drivesense.data.drowsiness import (
    extract_video_features, make_time_windows, label_from_video_stem, FEATURE_COLUMNS
)
from drivesense.features.landmarks import MediaPipeLandmarkExtractor
from drivesense.models.temporal import DrowsinessGRU, DrowsinessTemporalCNN
from drivesense.models.baseline import build_logistic_baseline, build_random_forest_baseline
from drivesense.data.splits import subject_independent_split, assert_no_subject_leakage

print("All required modules imported successfully.")

import subprocess
result = subprocess.run(
    ["pytest", "tests/test_drowsiness.py", "tests/test_landmarks.py", "tests/test_splits.py",
     "tests/test_models_temporal.py", "-q"],
    capture_output=True, text=True,
)
print(result.stdout[-3000:])
assert result.returncode == 0, "Test suite failed in this Colab environment - do not proceed until this passes."
print("Stage 1 OK: environment verified against the real test suite.")

## Stage 2 — Mount Google Drive for checkpointing

Colab's local disk is wiped on disconnect. Everything expensive (downloaded
videos, extracted features, trained checkpoints) is written to Drive as it's
produced, so a disconnect loses at most the current in-progress step, never
completed prior work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
CHECKPOINT_DIR = Path("/content/drive/MyDrive/DriveSense_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
(CHECKPOINT_DIR / "raw_drowsiness").mkdir(exist_ok=True)
(CHECKPOINT_DIR / "features").mkdir(exist_ok=True)
(CHECKPOINT_DIR / "models").mkdir(exist_ok=True)
(CHECKPOINT_DIR / "experiments").mkdir(exist_ok=True)
print("Checkpoint directory ready at", CHECKPOINT_DIR)

# Redirect the repo's data/raw/drowsiness location to Drive so
# scripts/uta_rldd_pipeline.py's downloads persist automatically, with no
# changes needed to that script.
import shutil
repo_raw_drowsiness = Path("data/raw/drowsiness")
if repo_raw_drowsiness.exists() and not repo_raw_drowsiness.is_symlink():
    shutil.rmtree(repo_raw_drowsiness)
if not repo_raw_drowsiness.exists():
    repo_raw_drowsiness.symlink_to(CHECKPOINT_DIR / "raw_drowsiness")
print("data/raw/drowsiness ->", repo_raw_drowsiness.resolve())

## Stage 3 — Kaggle authentication

Your own Kaggle account and token — never committed to this repo, never printed.

In [ ]:
from getpass import getpass
import os

kaggle_token = getpass("Kaggle API token (KAGGLE_API_TOKEN value, input hidden): ")
os.environ["KAGGLE_API_TOKEN"] = kaggle_token
del kaggle_token

import subprocess
result = subprocess.run(["kaggle", "datasets", "list", "-s", "uta-reallife-drowsiness", "--page-size", "1"],
                         capture_output=True, text=True)
print(result.stdout)
assert result.returncode == 0, "Kaggle authentication failed - check the token value."
print("Stage 3 OK: Kaggle authentication verified.")

## Stage 4 — Selective UTA-RLDD acquisition

Reuses the exact `scripts/uta_rldd_pipeline.py` already validated locally
against the real Kaggle file index. Colab's larger disk quota makes a bigger,
more statistically meaningful subject selection practical than the 6 used
for local pipeline development — but this still deliberately avoids pulling
the entire ~96.6GB mirror unless you have the Drive quota for it.

Adjust `N_SUBJECTS` below based on your available Google Drive storage
(check via Google Drive settings before running) — each subject is roughly
0.1-2GB (highly variable; `plan` below shows the real number for your
selection before downloading anything).

**Caveat:** `scripts/uta_rldd_pipeline.py fetch`'s built-in disk-space check (`shutil.disk_usage`) measures Colab's local ephemeral disk, not your Google Drive quota - but Stage 2 above redirects `data/raw/drowsiness` to a Drive-mounted symlink, so downloaded videos actually consume **Drive** space. The script's internal check is checking the wrong budget in this setup. Verify your real Google Drive free space yourself (Drive settings, or `!df -h /content/drive/MyDrive` after mounting) before choosing `N_SUBJECTS`, rather than trusting that check alone.

In [ ]:
N_SUBJECTS = 24  # adjust based on your real Drive quota - see the `plan` output below before committing

!python scripts/uta_rldd_pipeline.py index --force
!python scripts/uta_rldd_pipeline.py plan --n-subjects {N_SUBJECTS}

Review the real planned size printed above against your actual Drive free space, then choose the exact subject list to fetch (copy the `Suggested command` list from above, or pick your own).

In [ ]:
SUBJECTS = "45 17 31 16 27 44 04 35 18 21 39 41 42 49 50 51 52 56 61 64 66 72 75 81"  # EDIT to match the plan above

!python scripts/uta_rldd_pipeline.py fetch --subjects {SUBJECTS}

In [ ]:
# Real per-video validation (opens every file, reports actual fps/resolution/duration) -
# do not proceed to the expensive extraction stage on unvalidated video files.
!python scripts/uta_rldd_pipeline.py validate

import json
report = json.loads((Path("data/raw/drowsiness") / "_local_sample_validation.json").read_text())
n_failed = sum(1 for r in report if not r["opened"])
assert n_failed == 0, f"{n_failed} videos failed to open - fix/re-download before continuing."
print(f"Stage 4 OK: {len(report)} videos downloaded and validated.")

## Stage 5 — Real MediaPipe landmark extraction

**This is the stage that was blocked on the local machine.** Constructing
`MediaPipeLandmarkExtractor()` needs real RAM headroom that a desktop session
sharing ~7GB doesn't have; Colab's dedicated RAM should not have this
problem, but the cell below fails loudly with a clear message if it does
rather than silently producing empty results.

Checkpointing: each video's extracted features are saved to Drive
immediately after that video finishes, and already-completed videos are
skipped on re-run — a disconnect partway through loses at most the
currently-in-progress video, never completed ones.

In [ ]:
from drivesense.features.landmarks import MediaPipeLandmarkExtractor

try:
    extractor = MediaPipeLandmarkExtractor()
    print("MediaPipeLandmarkExtractor constructed successfully.")
except Exception as e:
    raise RuntimeError(
        "Failed to construct the real MediaPipe extractor even in Colab. "
        "This should not happen with a standard Colab runtime's RAM - check "
        "Runtime > View resources, and consider Runtime > Factory reset runtime "
        "if a previous cell left the session in a bad state."
    ) from e

In [ ]:
import pandas as pd
from pathlib import Path
from drivesense.data.drowsiness import extract_video_features, label_from_video_stem

RAW_DROWSINESS_DIR = Path("data/raw/drowsiness")
FEATURES_DIR = CHECKPOINT_DIR / "features"
SAMPLE_FPS = 5.0  # real samples per second, independent of each video's native fps

video_paths = sorted(RAW_DROWSINESS_DIR.glob("*/*.mp4")) + \
              sorted(RAW_DROWSINESS_DIR.glob("*/*.mov")) + \
              sorted(RAW_DROWSINESS_DIR.glob("*/*.MOV"))
assert video_paths, f"No videos found under {RAW_DROWSINESS_DIR} - did Stage 4 actually complete?"
print(f"Found {len(video_paths)} videos to process.")

failed_videos = []
for i, video_path in enumerate(video_paths, 1):
    subject = video_path.parent.name
    label = label_from_video_stem(video_path.stem)
    out_csv = FEATURES_DIR / f"{subject}_{label}.csv"

    if out_csv.exists():
        print(f"[{i}/{len(video_paths)}] SKIP (already extracted): {out_csv.name}")
        continue

    print(f"[{i}/{len(video_paths)}] Extracting {video_path} (subject={subject}, label={label})...")
    try:
        result = extract_video_features(
            video_path=video_path, extractor=extractor, subject=subject, label=label,
            sample_fps=SAMPLE_FPS,
        )
        df = pd.DataFrame(result.frame_rows)
        face_rate = result.n_faces_detected / max(result.n_frames_processed, 1)
        print(f"    {result.n_frames_processed} frames sampled, {face_rate:.1%} face-detected, "
              f"source fps={result.source_fps:.1f}")
        df.to_csv(out_csv, index=False)
    except Exception as e:
        print(f"    FAILED: {e}")
        failed_videos.append((str(video_path), str(e)))
        continue

if failed_videos:
    print(f"\n{len(failed_videos)} video(s) failed extraction - review before proceeding:")
    for path, err in failed_videos:
        print(f"  {path}: {err}")

extracted_csvs = sorted(FEATURES_DIR.glob("*.csv"))
print(f"\nStage 5: {len(extracted_csvs)} feature CSVs present in {FEATURES_DIR}")
assert len(extracted_csvs) > 0, "No features were successfully extracted - cannot proceed."


In [ ]:
# Combine all per-video CSVs into one real dataframe and checkpoint it.
combined = pd.concat([pd.read_csv(p) for p in extracted_csvs], ignore_index=True)
combined_path = FEATURES_DIR / "features_combined.csv"
combined.to_csv(combined_path, index=False)
print(f"Combined {len(extracted_csvs)} videos into {len(combined)} frame-rows -> {combined_path}")
print(combined.head())

## Stage 6 — Real EDA on extracted features

Actual statistics from the actual extracted data, not assumed.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("Total frame-rows:", len(combined))
print("Subjects:", combined['subject'].nunique())
print("\nRows per label:")
print(combined['label'].value_counts())
print("\nRows per subject:")
print(combined['subject'].value_counts())

face_rate_by_video = combined.groupby(['subject', 'label'])['face_detected'].mean()
print("\nFace-detection rate per video (subject, label):")
print(face_rate_by_video)
low_face_rate = face_rate_by_video[face_rate_by_video < 0.5]
if len(low_face_rate) > 0:
    print(f"\nWARNING: {len(low_face_rate)} video(s) have <50% face-detection rate - "
          "these will mostly be dropped by the face-fraction filter in windowing:")
    print(low_face_rate)

print("\nEAR/MAR descriptive stats (face-detected frames only):")
print(combined[combined['face_detected']][['left_ear', 'right_ear', 'mar']].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for label in combined['label'].unique():
    subset = combined[(combined['label'] == label) & combined['face_detected']]
    axes[0].hist(subset['left_ear'], bins=30, alpha=0.5, label=label)
axes[0].set_title("Left EAR distribution by label (real data)")
axes[0].set_xlabel("EAR")
axes[0].legend()

face_rate_by_video.plot(kind='bar', ax=axes[1])
axes[1].set_title("Face-detection rate per video")
axes[1].set_ylabel("fraction of frames with a detected face")
fig.tight_layout()

plots_dir = CHECKPOINT_DIR / "experiments" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(plots_dir / "drowsiness_eda.png", dpi=120)
print(f"\nSaved real EDA plot to {plots_dir / 'drowsiness_eda.png'}")

eda_summary = {
    "total_frames": int(len(combined)),
    "n_subjects": int(combined['subject'].nunique()),
    "rows_per_label": combined['label'].value_counts().to_dict(),
    "overall_face_detection_rate": float(combined['face_detected'].mean()),
    "face_detection_rate_by_video": {f"{s}_{l}": float(v) for (s, l), v in face_rate_by_video.items()},
}
import json
(CHECKPOINT_DIR / "experiments" / "drowsiness_eda_summary.json").write_text(json.dumps(eda_summary, indent=2))
print("Stage 6 OK: real EDA complete.")

## Stage 7 — Time-based windowing

Uses `make_time_windows` (already unit-tested locally with synthetic
multi-fps data — see `tests/test_drowsiness.py`) to slice the real extracted
features into fixed-duration windows, resampled to a fixed number of time
steps regardless of each video's native frame rate. This is what makes a
"10-second window" actually mean 10 real seconds for every subject, whether
their video was recorded at 12fps or 30fps.

`WINDOW_SECONDS=10, STRIDE_SECONDS=5, TARGET_STEPS=30` is a reasonable
starting choice (10s is enough to capture a blink/yawn pattern; 30 steps
across 10s ≈ 3Hz effective resampled rate) — adjust and re-run this stage
if the resulting windows-per-class balance looks unreasonable in Stage 8.

In [ ]:
from drivesense.data.drowsiness import make_time_windows, FEATURE_COLUMNS
import numpy as np

WINDOW_SECONDS = 10.0
STRIDE_SECONDS = 5.0
TARGET_STEPS = 30
REQUIRE_FACE_FRACTION = 0.5

windows, labels, subjects = make_time_windows(
    combined, window_seconds=WINDOW_SECONDS, stride_seconds=STRIDE_SECONDS,
    target_steps=TARGET_STEPS, require_face_fraction=REQUIRE_FACE_FRACTION,
)
assert len(windows) > 0, "No windows produced - check WINDOW_SECONDS/face-detection rates above."
print(f"Produced {len(windows)} windows of shape {windows.shape[1:]} from {combined['subject'].nunique()} subjects")
print("Windows per label:")
import collections
print(collections.Counter(labels))
print("Windows per subject:")
print(collections.Counter(subjects))

windows_path = CHECKPOINT_DIR / "features" / "windows.npz"
np.savez_compressed(windows_path, windows=windows, labels=labels, subjects=subjects)
print(f"Stage 7 OK: checkpointed windows to {windows_path}")

## Stage 8 — Strict subject-independent train/val/test split

Uses the same `subject_independent_split` already used (and tested) for the
distraction model, so a subject's windows can never appear in more than one
split. Explicit assertions below re-verify zero overlap independently of
what the function itself already asserts internally, and print the exact
subject lists per split so this is defensible in a defense/report.


In [ ]:

import pandas as pd

meta_df = pd.DataFrame({"subject": subjects, "label": labels, "window_idx": range(len(windows))})

train_meta, val_meta, test_meta = subject_independent_split(
    meta_df, subject_col="subject", test_size=0.2, val_size=0.15, random_state=42,
)

train_subj, val_subj, test_subj = set(train_meta["subject"]), set(val_meta["subject"]), set(test_meta["subject"])
assert not (train_subj & val_subj), "train/val subject leakage"
assert not (train_subj & test_subj), "train/test subject leakage"
assert not (val_subj & test_subj), "val/test subject leakage"

print(f"TRAIN subjects ({len(train_subj)}): {sorted(train_subj)}")
print(f"VAL   subjects ({len(val_subj)}): {sorted(val_subj)}")
print(f"TEST  subjects ({len(test_subj)}): {sorted(test_subj)}")
print(f"TRAIN windows: {len(train_meta)}, VAL windows: {len(val_meta)}, TEST windows: {len(test_meta)}")
print("Leakage assertions passed: no subject appears in more than one split.")

X_train, y_train = windows[train_meta["window_idx"].to_numpy()], labels[train_meta["window_idx"].to_numpy()]
X_val, y_val = windows[val_meta["window_idx"].to_numpy()], labels[val_meta["window_idx"].to_numpy()]
X_test, y_test = windows[test_meta["window_idx"].to_numpy()], labels[test_meta["window_idx"].to_numpy()]

LABEL_ORDER = ["alert", "low_vigilant", "drowsy"]
label_to_idx = {l: i for i, l in enumerate(LABEL_ORDER)}
y_train_idx = np.array([label_to_idx[l] for l in y_train])
y_val_idx = np.array([label_to_idx[l] for l in y_val])
y_test_idx = np.array([label_to_idx[l] for l in y_test])

split_record = {
    "train_subjects": sorted(train_subj), "val_subjects": sorted(val_subj), "test_subjects": sorted(test_subj),
    "train_windows": int(len(train_meta)), "val_windows": int(len(val_meta)), "test_windows": int(len(test_meta)),
}
import json
(CHECKPOINT_DIR / "experiments" / "drowsiness_split.json").write_text(json.dumps(split_record, indent=2))
print("Stage 8 OK: split recorded to Drive.")


## Stage 9 — Three real experiments

This project's primary research question is whether temporal modeling
improves driver-state recognition over single-frame classification, under
subject-independent evaluation. Three experiments, all on the identical
split above:

1. **Non-temporal baseline** — mean-pools each window's 6 features across
   time (discarding all temporal information) and feeds that single vector
   into a `LogisticRegression` / `RandomForestClassifier`
   (`src/drivesense/models/baseline.py`) — an interpretable, non-temporal
   reference point.
2. **GRU** (`DrowsinessGRU`) — sees the full time sequence within each
   window.
3. **1D-CNN** (`DrowsinessTemporalCNN`) — sees the full time sequence via
   1D convolutions instead of recurrence.

Do not assume temporal modeling wins — that's the actual question this
comparison answers with real numbers.

In [ ]:

import torch, torch.nn as nn, time, json
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import subprocess

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", DEVICE)

def git_commit_hash():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return "unknown"

def train_torch_model(model, X_train, y_train, X_val, y_val, epochs=30, lr=1e-3,
                       batch_size=32, patience=5, run_id="run", checkpoint_path=None):
    """Trains with mid-run checkpointing: every new best-val epoch is written
    to `checkpoint_path` immediately (not just returned at the end), so a
    Colab disconnect mid-training loses at most the epochs since the last
    improvement, never the whole run. If `checkpoint_path` already exists
    (e.g. resuming after a disconnect), loads it and reports the existing
    best val macro F1 instead of retraining from scratch - delete the file
    manually if you actually want to retrain this model."""
    model = model.to(DEVICE)

    if checkpoint_path is not None and Path(checkpoint_path).exists():
        ckpt = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        print(f"[{run_id}] Found existing checkpoint at {checkpoint_path} "
              f"(val_macro_f1={ckpt['val_macro_f1']:.4f}) - skipping training. "
              "Delete this file first if you want to retrain.")
        return model, ckpt.get("history", []), ckpt["val_macro_f1"]

    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = []
    best_val_f1 = -1.0
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        train_preds, train_labels = [], []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
            train_preds.extend(out.argmax(1).cpu().tolist())
            train_labels.extend(yb.cpu().tolist())
        train_loss /= len(train_ds)
        train_f1 = f1_score(train_labels, train_preds, average="macro", zero_division=0)

        model.eval()
        val_loss = 0.0
        val_preds, val_labels = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb)
                loss = criterion(out, yb)
                val_loss += loss.item() * xb.size(0)
                val_preds.extend(out.argmax(1).cpu().tolist())
                val_labels.extend(yb.cpu().tolist())
        val_loss /= len(val_ds)
        val_acc = accuracy_score(val_labels, val_preds)
        val_f1 = f1_score(val_labels, val_preds, average="macro", zero_division=0)
        epoch_time = time.time() - t0

        print(f"[{run_id}] Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} train_f1={train_f1:.4f} "
              f"| val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f} | {epoch_time:.1f}s")
        history.append({"epoch": epoch, "train_loss": train_loss, "train_macro_f1": train_f1,
                         "val_loss": val_loss, "val_acc": val_acc, "val_macro_f1": val_f1,
                         "epoch_time_sec": epoch_time})

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_without_improvement = 0
            if checkpoint_path is not None:
                torch.save({"model_state_dict": model.state_dict(), "val_macro_f1": best_val_f1,
                            "history": history, "epoch": epoch}, checkpoint_path)
                print(f"[{run_id}]   -> new best (val_macro_f1={best_val_f1:.4f}), checkpointed to {checkpoint_path}")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"[{run_id}] Early stopping: no improvement for {patience} epochs.")
                break

    if checkpoint_path is not None:
        best = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(best["model_state_dict"])
    return model, history, best_val_f1

print("Training helper defined (with mid-run checkpointing and resume-on-rerun).")


### Experiment A — non-temporal baseline (mean-pooled features)

In [ ]:

from sklearn.metrics import f1_score as sk_f1

X_train_pooled = X_train.mean(axis=1)  # (n, target_steps, 6) -> (n, 6), discards all temporal info
X_val_pooled = X_val.mean(axis=1)
X_test_pooled = X_test.mean(axis=1)

baseline_model = build_random_forest_baseline(random_state=42)
baseline_model.fit(X_train_pooled, y_train_idx)
baseline_val_preds = baseline_model.predict(X_val_pooled)
baseline_val_f1 = sk_f1(y_val_idx, baseline_val_preds, average="macro", zero_division=0)
print(f"Baseline (RandomForest, mean-pooled, non-temporal) val macro F1: {baseline_val_f1:.4f}")

RUN_ID_BASELINE = f"drowsiness_baseline_{time.strftime('%Y%m%d_%H%M%S')}"
import joblib
baseline_ckpt = CHECKPOINT_DIR / "models" / f"{RUN_ID_BASELINE}.joblib"
joblib.dump(baseline_model, baseline_ckpt)
print(f"Saved baseline checkpoint to {baseline_ckpt}")


### Experiment B — GRU (full time sequence)

In [ ]:

# Fixed (non-timestamped) run_id so a re-run after a Colab disconnect finds
# the same checkpoint path and resumes instead of starting a fresh timestamped
# run that can't be matched back up. If you deliberately want to retrain from
# scratch, delete the .pt file at gru_ckpt first.
RUN_ID_GRU = "drowsiness_gru"
gru_ckpt = CHECKPOINT_DIR / "models" / f"{RUN_ID_GRU}.pt"
gru_model = DrowsinessGRU(input_dim=len(FEATURE_COLUMNS), num_classes=3, hidden_dim=64, dropout=0.3)
gru_model, gru_history, gru_val_f1 = train_torch_model(
    gru_model, X_train, y_train_idx, X_val, y_val_idx, epochs=30, lr=1e-3,
    run_id=RUN_ID_GRU, checkpoint_path=gru_ckpt,
)
# Merge architecture metadata into the checkpoint (train_torch_model only
# knows about weights/history, not the model's constructor arguments).
ckpt_data = torch.load(gru_ckpt, map_location=DEVICE, weights_only=False)
ckpt_data.update({"model_name": "gru", "input_dim": len(FEATURE_COLUMNS), "num_classes": 3})
torch.save(ckpt_data, gru_ckpt)
print(f"Best GRU val macro F1: {gru_val_f1:.4f} -> {gru_ckpt}")


### Experiment C — 1D-CNN (full time sequence)

In [ ]:

# Fixed (non-timestamped) run_id - see the GRU cell above for why.
RUN_ID_CNN = "drowsiness_cnn1d"
cnn_ckpt = CHECKPOINT_DIR / "models" / f"{RUN_ID_CNN}.pt"
cnn_model = DrowsinessTemporalCNN(input_dim=len(FEATURE_COLUMNS), num_classes=3, hidden_channels=32)
cnn_model, cnn_history, cnn_val_f1 = train_torch_model(
    cnn_model, X_train, y_train_idx, X_val, y_val_idx, epochs=30, lr=1e-3,
    run_id=RUN_ID_CNN, checkpoint_path=cnn_ckpt,
)
ckpt_data = torch.load(cnn_ckpt, map_location=DEVICE, weights_only=False)
ckpt_data.update({"model_name": "temporal_cnn", "input_dim": len(FEATURE_COLUMNS), "num_classes": 3})
torch.save(ckpt_data, cnn_ckpt)
print(f"Best 1D-CNN val macro F1: {cnn_val_f1:.4f} -> {cnn_ckpt}")


## Stage 10 — Model selection (VALIDATION ONLY) and final test evaluation

Per the Brief and this project's own established convention (see the
distraction model's `docs/LEARNING_LOG.md` entry on this exact point): the
model is selected using **validation** macro F1 only. The test set is
touched exactly once, below, for the selected model only — not used to
compare all three candidates.

In [ ]:

candidates = {
    "baseline_random_forest": baseline_val_f1,
    "gru": gru_val_f1,
    "temporal_cnn": cnn_val_f1,
}
print("Validation macro F1 per candidate (selection signal):")
for name, f1 in candidates.items():
    print(f"  {name}: {f1:.4f}")

SELECTED_MODEL = max(candidates, key=candidates.get)
print(f"\nSelected model (by validation macro F1 only): {SELECTED_MODEL}")


In [ ]:

import numpy as np
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_recall_fscore_support, confusion_matrix,
)

def evaluate_torch_model(model, X, y_idx, device):
    model.eval()
    preds = []
    latencies = []
    with torch.no_grad():
        for i in range(len(X)):
            xb = torch.tensor(X[i:i+1], dtype=torch.float32).to(device)
            t0 = time.time()
            out = model(xb)
            latencies.append(time.time() - t0)
            preds.append(int(out.argmax(1).item()))
    return np.array(preds), latencies

if SELECTED_MODEL == "baseline_random_forest":
    test_preds = baseline_model.predict(X_test_pooled)
    latencies = None
elif SELECTED_MODEL == "gru":
    test_preds, latencies = evaluate_torch_model(gru_model, X_test, y_test_idx, DEVICE)
else:
    test_preds, latencies = evaluate_torch_model(cnn_model, X_test, y_test_idx, DEVICE)

acc = accuracy_score(y_test_idx, test_preds)
bal_acc = balanced_accuracy_score(y_test_idx, test_preds)
macro_f1 = f1_score(y_test_idx, test_preds, average="macro", zero_division=0)
weighted_f1 = f1_score(y_test_idx, test_preds, average="weighted", zero_division=0)
precision, recall, f1_per_class, support = precision_recall_fscore_support(
    y_test_idx, test_preds, labels=list(range(3)), zero_division=0
)
cm = confusion_matrix(y_test_idx, test_preds, labels=list(range(3)))

print(f"REAL, ONCE-ONLY TEST RESULTS for {SELECTED_MODEL}:")
print(f"  accuracy: {acc:.4f}")
print(f"  balanced_accuracy: {bal_acc:.4f}")
print(f"  macro_f1: {macro_f1:.4f}")
print(f"  weighted_f1: {weighted_f1:.4f}")
for i, lbl in enumerate(LABEL_ORDER):
    print(f"  {lbl}: precision={precision[i]:.3f} recall={recall[i]:.3f} f1={f1_per_class[i]:.3f} support={support[i]}")
print("  confusion_matrix (rows=true, cols=pred, order=alert/low_vigilant/drowsy):")
print(cm)
if latencies:
    avg_fps = 1.0 / (sum(latencies) / len(latencies))
    print(f"  inference: {avg_fps:.1f} windows/sec ({DEVICE})")
else:
    avg_fps = None

test_results = {
    "selected_model": SELECTED_MODEL,
    "validation_macro_f1_per_candidate": candidates,
    "test_accuracy": float(acc), "test_balanced_accuracy": float(bal_acc),
    "test_macro_f1": float(macro_f1), "test_weighted_f1": float(weighted_f1),
    "per_class": {LABEL_ORDER[i]: {"precision": float(precision[i]), "recall": float(recall[i]),
                                     "f1": float(f1_per_class[i]), "support": int(support[i])}
                  for i in range(3)},
    "confusion_matrix": cm.tolist(),
    "inference_fps": avg_fps,
    "window_config": {"window_seconds": WINDOW_SECONDS, "stride_seconds": STRIDE_SECONDS,
                       "target_steps": TARGET_STEPS, "sample_fps": SAMPLE_FPS},
    "split": split_record,
    "git_commit": git_commit_hash(),
}
(CHECKPOINT_DIR / "experiments" / "drowsiness_test_results.json").write_text(json.dumps(test_results, indent=2))
print("\nStage 10 OK: real test results saved.")


## Stage 11 — Plots and tables for the report/defense

Real training curves and a real confusion-matrix heatmap from the actual selected run above.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

selected_history = {"gru": gru_history, "temporal_cnn": cnn_history}.get(SELECTED_MODEL)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
if selected_history is not None:
    epochs = [h["epoch"] for h in selected_history]
    axes[0].plot(epochs, [h["train_loss"] for h in selected_history], label="train loss")
    axes[0].plot(epochs, [h["val_loss"] for h in selected_history], label="val loss")
    axes[0].set_title(f"{SELECTED_MODEL}: real training curves")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, "Selected model has no per-epoch curve\n(non-temporal baseline)",
                  ha="center", va="center")
    axes[0].set_axis_off()

im = axes[1].imshow(cm, cmap="Blues")
axes[1].set_xticks(range(3)); axes[1].set_xticklabels(LABEL_ORDER, rotation=45)
axes[1].set_yticks(range(3)); axes[1].set_yticklabels(LABEL_ORDER)
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
axes[1].set_title(f"{SELECTED_MODEL}: real test confusion matrix")
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, str(cm[i, j]), ha="center", va="center",
                      color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=axes[1], fraction=0.046)
fig.tight_layout()

plots_dir = CHECKPOINT_DIR / "experiments" / "plots"
fig.savefig(plots_dir / f"{SELECTED_MODEL}_results.png", dpi=120)
print(f"Saved real results plot to {plots_dir / (SELECTED_MODEL + '_results.png')}")

per_class_table = "| Class | Precision | Recall | F1 | Support |\n|---|---|---|---|---|\n"
for i, lbl in enumerate(LABEL_ORDER):
    per_class_table += f"| {lbl} | {precision[i]:.3f} | {recall[i]:.3f} | {f1_per_class[i]:.3f} | {support[i]} |\n"
print(per_class_table)
(CHECKPOINT_DIR / "experiments" / f"{SELECTED_MODEL}_per_class_table.md").write_text(per_class_table)
print("Stage 11 OK: plots and tables saved to Drive.")


## Stage 12 — Export the final checkpoint/config and transfer results back

Two separate things happen here, deliberately kept apart:

1. **Small, real result artifacts** (JSON metrics, markdown tables, PNG
   plots — no raw video, no model weights) are committed and pushed
   directly back to this same GitHub repo, the same way the distraction
   work already in this repo was committed — so `docs/Master_Plan_Status.md`
   etc. can be updated with these real numbers afterward, from either
   Colab or your local machine.
2. **The actual trained checkpoint** (a binary `.pt`/`.joblib` file) is
   never committed to git (matches `.gitignore` — model artifacts are
   regenerable, not source). It's already safe in Google Drive
   (`CHECKPOINT_DIR / "models"`); the cell below also offers a direct
   browser download so you can place it under `models/drowsiness/` on your
   local machine for the demo/inference integration.

In [ ]:

preprocessing_config = {
    "feature_columns": FEATURE_COLUMNS,
    "label_order": LABEL_ORDER,
    "window_seconds": WINDOW_SECONDS,
    "stride_seconds": STRIDE_SECONDS,
    "target_steps": TARGET_STEPS,
    "sample_fps": SAMPLE_FPS,
    "selected_model": SELECTED_MODEL,
    "git_commit_used": git_commit_hash(),
}
config_path = CHECKPOINT_DIR / "experiments" / "drowsiness_preprocessing_config.json"
config_path.write_text(json.dumps(preprocessing_config, indent=2))
print(f"Saved preprocessing config to {config_path}")
print(json.dumps(preprocessing_config, indent=2))


In [ ]:

# Commit only small, real result files - never data/checkpoints (already
# excluded by .gitignore, double-checked explicitly below as a safety net).
import shutil

repo_experiments_dir = Path("experiments/drowsiness")
repo_experiments_dir.mkdir(parents=True, exist_ok=True)
for fname in ["drowsiness_eda_summary.json", "drowsiness_split.json", "drowsiness_test_results.json",
              "drowsiness_preprocessing_config.json", f"{SELECTED_MODEL}_per_class_table.md"]:
    src = CHECKPOINT_DIR / "experiments" / fname
    if src.exists():
        shutil.copy(src, repo_experiments_dir / fname)

print("Files staged for commit:")
!git status --short

# Safety net: refuse to proceed if anything under data/ or models/ or a
# credential-shaped file is about to be committed.
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout
forbidden = [l for l in status.splitlines() if any(
    p in l for p in ["data/raw", "data/processed", "models/", ".pt", ".joblib", "kaggle.json", ".zip"]
)]
assert not forbidden, f"Refusing to commit - forbidden paths staged: {forbidden}"

!git add experiments/drowsiness/
!git commit -m "Add real drowsiness experiment results from Colab run (see notebook 02)"
!git push origin main
print("Stage 12 OK: real result artifacts pushed to GitHub.")


In [ ]:

# Download the actual trained checkpoint to place locally under models/drowsiness/.
from google.colab import files

if SELECTED_MODEL == "baseline_random_forest":
    files.download(str(baseline_ckpt))
elif SELECTED_MODEL == "gru":
    files.download(str(gru_ckpt))
else:
    files.download(str(cnn_ckpt))

print("Checkpoint download triggered - save it to models/drowsiness/ in your local DriveSense clone.")
print("It is also safe in Google Drive at:", CHECKPOINT_DIR / "models")


## Done — but read this before reporting any number from this notebook

Everything above is **real, working code**, built and locally validated
(syntax-checked cell by cell, and every non-MediaPipe function it calls —
`extract_video_features`, `make_time_windows`, the split logic — is already
unit-tested against synthetic data in `tests/test_drowsiness.py`,
`tests/test_splits.py`). **None of it has been executed against real UTA-RLDD
video yet.** The moment you run Stages 0-12 in order in an actual Colab
runtime, every number becomes real.

After a real run:
1. Copy the real numbers from `experiments/drowsiness/drowsiness_test_results.json`
   (now committed to this repo) into `README.md`, `docs/Master_Plan_Status.md`,
   and `docs/LEARNING_LOG.md` — using the exact same "what/why/how/result/what to
   say to the professor" format already used for the distraction model.
2. Place the downloaded checkpoint under `models/drowsiness/` locally.
3. Wire it into `scripts/demo_distraction.py`'s risk-engine pattern (currently
   distraction-only, `drowsiness_label=None`) to get real temporal risk
   fusion for the first time.

Never copy a number out of this notebook that you did not personally
generate by running these cells.